# Day 8: Snowflake Storage Integration & Dynamic Tables
This notebook uses DuckDB to simulate incremental query behavior and SQL transformation logic similar to Snowflake Dynamic Tables.

In [ ]:
!pip install duckdb

In [ ]:
import duckdb

# Initialize DuckDB memory database
con = duckdb.connect(':memory:')

# Create base Silver table (Mocking Iceberg table in Snowflake)
con.execute("""
CREATE TABLE silver_condition_occurrence (
    person_id INT,
    condition_concept_id INT,
    condition_start_date DATE
)
""")

# Insert initial data
con.execute("""
INSERT INTO silver_condition_occurrence VALUES 
(1, 101, CURRENT_DATE - INTERVAL 5 DAY),
(1, 102, CURRENT_DATE - INTERVAL 40 DAY),
(2, 201, CURRENT_DATE - INTERVAL 10 DAY)
""")

print("Base Table (silver_condition_occurrence):")
print(con.execute("SELECT * FROM silver_condition_occurrence").df())

# Simulate Dynamic Table: 5 min lag for recent conditions (< 30 days)
con.execute("""
CREATE VIEW dt_silver_recent_conditions AS
SELECT person_id, condition_concept_id, condition_start_date
FROM silver_condition_occurrence
WHERE condition_start_date >= CURRENT_DATE - INTERVAL 30 DAY
""")

print("\nSimulated Dynamic Table (dt_silver_recent_conditions):")
print(con.execute("SELECT * FROM dt_silver_recent_conditions").df())

# Insert new data (Simulating continuous ingestion)
con.execute("""
INSERT INTO silver_condition_occurrence VALUES (3, 301, CURRENT_DATE - INTERVAL 1 DAY)
""")

print("\nSimulated Dynamic Table after new data arrival:")
print(con.execute("SELECT * FROM dt_silver_recent_conditions").df())